# wrap-forward-fn-generic — faded example 1: The unbox step of the wrapper

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wrap-forward-fn-generic`. The last cell reports your progress on the `Backprop: wrap forward fn` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: wrap forward fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wrap-forward-fn-generic`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wrap-forward-fn-generic"
DD_SUBTOPIC = "Backprop: wrap forward fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Inside `tensor_func`, the unbox step replaces every `Tensor` arg with its `.array` so the raw `fwd_fn` never sees a wrapper. Non-Tensor args pass through. This is the first of the unbox → call → box trio.

## Faded exercise 1

### Unbox the args before calling

Implement `wrap_forward_fn(fwd_fn)`. The call and box steps are written; complete the unbox line that turns `args` into `raw_args` (Tensors to `.array`, everything else unchanged).

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = None  # TODO: fill in this step — read the prompt cell above
        out_raw = fwd_fn(*raw_args, **kwargs)
        return Tensor(out_raw)
    return tensor_func

neg = wrap_forward_fn(t.neg)
print(neg(Tensor([1.0, -2.0])).array.tolist())


def _test():
    add = wrap_forward_fn(t.add)
    a = Tensor([1.0, 2.0, 3.0])
    b = Tensor([10.0, 20.0, 30.0])
    out = add(a, b)
    assert isinstance(out, Tensor)
    # independent truth
    assert t.allclose(out.array, t.tensor([11.0, 22.0, 33.0]))
    # scalar arg passes through unboxed: t.add(tensor, scalar)
    out2 = add(a, 5)
    assert t.allclose(out2.array, t.tensor([6.0, 7.0, 8.0]))


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class Tensor:
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        return Tensor(out_raw)
    return tensor_func

neg = wrap_forward_fn(t.neg)
print(neg(Tensor([1.0, -2.0])).array.tolist())
```
</details>